# Duplicates, Outliers, and Data Types

Detect repeated records, parse types, and inspect unusual numeric values with context.

> 🧭 This notebook focuses on practical data work. Examples are intentionally compact and do not cover machine learning.

## Learning goals
- Standardize text before checking duplicates.
- Parse numeric and date columns with explicit error handling.
- Flag outliers with the IQR rule and inspect them before removing anything.
- Keep a raw copy and document each exclusion.

In [1]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "record_id": [1, 2, 2, 3, 4, 5],
    "district": ["Central", " north ", "North", "WEST", "Central", "south"],
    "amount": [25, 30, 30, 28, 29, 900],
    "date": ["2025-01-01", "2025-01-02", "2025-01-02", "bad-date", "2025-01-04", "2025-01-05"],
})
raw = df.copy()
df["district"] = df["district"].str.strip().str.title()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
display(df)
print(df.dtypes)

record_id             int64
district             object
amount                int64
date         datetime64[ns]
dtype: object


,record_id,district,amount,date
0,1,Central,25,2025-01-01
1,2,North,30,2025-01-02
2,2,North,30,2025-01-02
3,3,West,28,NaT
4,4,Central,29,2025-01-04
5,5,South,900,2025-01-05


## Duplicate records
A duplicate can be an exact duplicate or a duplicate according to business keys. Two rows may share a customer ID but represent two legitimate transactions, so choose the key based on the unit of observation.

In [2]:
print("Exact duplicates:", df.duplicated().sum())
print("Repeated record IDs:")
display(df.loc[df.duplicated(subset=["record_id"], keep=False)].sort_values("record_id"))

# Example rule: retain the first record per record_id. Confirm this rule is valid before production use.
deduplicated = df.drop_duplicates(subset=["record_id"], keep="first")
display(deduplicated)

Exact duplicates: 1
Repeated record IDs:


,record_id,district,amount,date
1,2,North,30,2025-01-02
2,2,North,30,2025-01-02


,record_id,district,amount,date
0,1,Central,25,2025-01-01
1,2,North,30,2025-01-02
3,3,West,28,NaT
4,4,Central,29,2025-01-04
5,5,South,900,2025-01-05


## IQR outlier flags
For numeric data, the IQR rule flags values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR. It is a screening heuristic, not proof that a value is erroneous. Skewed distributions may naturally contain extreme values.

In [3]:
q1 = df["amount"].quantile(0.25)
q3 = df["amount"].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df["amount_outlier_flag"] = ~df["amount"].between(lower, upper)
print({"q1": q1, "q3": q3, "lower_bound": lower, "upper_bound": upper})
display(df[["record_id", "amount", "amount_outlier_flag"]])

{'q1': np.float64(28.25), 'q3': np.float64(30.0), 'lower_bound': np.float64(25.625), 'upper_bound': np.float64(32.625)}


,record_id,amount,amount_outlier_flag
0,1,25,True
1,2,30,False
2,2,30,False
3,3,28,False
4,4,29,False
5,5,900,True


## Safer numeric parsing
`errors="coerce"` turns invalid strings into missing values. Always count how many values failed conversion so corrupt values do not silently disappear.

In [4]:
raw_amounts = pd.Series(["12.5", "8", "unknown", " 7.1 ", None])
parsed = pd.to_numeric(raw_amounts, errors="coerce")
print(parsed)
print("Invalid non-empty values:", (raw_amounts.notna() & parsed.isna()).sum())

# Use validation rules based on the domain, not only distributional thresholds.
valid_amount = parsed.ge(0) | parsed.isna()
print("All values non-negative or missing?", bool(valid_amount.all()))

0    12.5
1     8.0
2     NaN
3     7.1
4     NaN
dtype: float64
Invalid non-empty values: 1
All values non-negative or missing? True


## Practice
Normalize district labels, parse dates, and flag amounts above the 95th percentile without automatically deleting them.

In [5]:
cleaned = raw.copy()
cleaned["district"] = cleaned["district"].str.strip().str.title()
cleaned["date"] = pd.to_datetime(cleaned["date"], errors="coerce")
threshold = cleaned["amount"].quantile(0.95)
cleaned["above_p95"] = cleaned["amount"] > threshold
display(cleaned)
print("Date parse failures:", cleaned["date"].isna().sum())

Date parse failures: 1


,record_id,district,amount,date,above_p95
0,1,Central,25,2025-01-01,False
1,2,North,30,2025-01-02,False
2,2,North,30,2025-01-02,False
3,3,West,28,NaT,False
4,4,Central,29,2025-01-04,False
5,5,South,900,2025-01-05,True
